# Data validation, cleaning and merging

**Goal:** one daily table that the analysis can trust: right values, and each value
used only once it was public.

**Pipeline**

`src/data/*.py` (retrieve + check) → `tests/` (validate) → `src/cleaning.py` (merge)
→ `data/merged_daily.parquet`

All choices and their reasons are logged in `docs/decisions.md`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src import config
from src.cleaning import OUTPUT_FILE, load_sources

sources = load_sources()
daily = pd.read_parquet(OUTPUT_FILE)

# Figure style: recessive axes, categorical slots 1-2
BLUE, ORANGE, MUTED = "#2a78d6", "#eb6834", "#5f5e57"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.grid": True, "grid.color": "#e6e5e0",
    "grid.linewidth": 0.6, "lines.linewidth": 2, "figure.figsize": (10, 4),
})

## 1. Five sources, four frequencies

| Source | Data | Frequency | Index (date the value refers to) | Public from (`release_date`) |
|--------|------|-----------|----------------------------------|------------------------------|
| Yahoo Finance | USD/JPY, equities, Bitcoin, USD/MXN, gold, Brent, VIX | daily | trading day | same day |
| FRED (OECD) | US and Japan 3-month interbank rates | monthly | month start | + 3 months |
| CFTC | speculative positioning in yen futures | weekly | report date (Tuesday) | next Friday |
| BIS | yen credit to borrowers outside Japan | quarterly | quarter end | + 4 months |
| MoF Japan | FX interventions | event | intervention day | quarter end + 45 days |

**Window:** 2000-01-01 to 2026-09-30, fixed so results do not change between runs.
Carry tests start in 2002-04, the first month with a Japanese rate.

## 2. Validation: three layers

| Layer | Question | How | Examples |
|-------|----------|-----|----------|
| **Integrity** | Is the file what we think it is? | checks on download that stop the code | BIS units (JPY, millions); FRED series present, monthly, rates within −2%…25%; MoF file layout |
| **Content** | Are the values plausible and consistent? | automated tests on saved files | MoF days add up to MoF's quarterly totals; known July 2024 CFTC peak; no impossible jumps |
| **Timing** | When could an investor have known each value? | release rules checked against real releases | see below |

Timing is part of validation: correct data used **before it was published** gives
results that look better than reality (look-ahead bias).

### Timing checks

| Rule | Evidence | Verdict |
|------|----------|---------|
| FRED: month start + 3 months | 15 Oct 2026 update: US data to September, Japan to August (≈1.5 and 2.5 months) | safe |
| BIS: quarter end + 4 months | 2026-Q2 released on 29 Oct 2026; rule gives 30 Oct | safe (one day late) |
| MoF: quarter end + 45 days | on or after every quarterly release checked (e.g. Q3 2024 on 8 Nov) | safe |
| CFTC: next Friday | reports delayed by the 2013 and 2018–19 US shutdowns flagged until CFTC was back on schedule (8 Nov 2013, 8 Mar 2019) | delayed reports excluded |

### A break found by validation: the Japanese rate in 2025

In January 2025 FRED's Japanese 3-month rate jumps by 38bp, more than the BoJ's 25bp hike
that month. Compared with the official JBA TIBOR (daily PDFs from the Japanese Bankers
Association):

| Period | FRED vs month-end 3-month JBA TIBOR |
|--------|---------------------------------------|
| 2025 | **identical**, every month |
| 2024 | 6–23bp lower, every month |

The source rate changed. We keep the series, mark 2025-01 as a break, and check the
results without that month's change.

In [ ]:
jp = sources["fred"]["jp_3m"]
pd.DataFrame({"jp_3m (%)": jp, "change (pp)": jp.diff()}).loc["2024-10-01":"2025-03-01"].round(3)

## 3. Cleaning: one rule for every source

1. **Index = the date the value refers to**, with `release_date` as a column.
   Indexing by release date fails for MoF: all intervention days of a quarter share one
   release date.
2. **Each analysis picks its date:**
   - event study → the event date (markets react on the day);
   - predictive tests (H2, H3) → only values already public (`_known` columns);
   - carry accrual → the rate of the day's own month.
3. **No filling of prices:** a closed market stays empty, to avoid fake zero returns.
4. **Bad prints removed in USD/JPY only** (four days in 2008): a one-day jump of more
   than 3% that reverses the next day. Not applied to VIX, Bitcoin or oil, where such
   moves are real.

## 4. Merging: one row per USD/JPY trading day

| Columns | Meaning on day *d* | Used for |
|---------|--------------------|----------|
| `usdjpy` … `vix` | closing price, empty if that market was closed | returns |
| `us_3m`, `jp_3m`, `rate_diff` | rate of the month containing *d* | carry accrual |
| `*_known` (rates, CFTC, BIS) | latest value released **before** *d* | predictive tests |
| `intervention_bn_jpy` | MoF intervention on *d* (> 0 = yen bought), else 0 | event study |

**No look-ahead by construction:** a value is used from the trading day *after* its
release date. A CFTC report published on Friday afternoon enters on Monday, so a
Friday decision uses the previous week's report.

In [ ]:
print(f"{len(daily):,} trading days, {daily.index[0].date()} to {daily.index[-1].date()}, "
      f"{daily.shape[1]} columns")

In [ ]:
# Event date vs release date: the July 2024 interventions
daily.loc["2024-07-09":"2024-07-16", ["usdjpy", "intervention_bn_jpy"]].round(2)

In [ ]:
window = daily.loc["2022-01-01":"2024-12-31"]
bought = window[window["intervention_bn_jpy"] > 0]

fig, ax = plt.subplots()
ax.plot(window.index, window["usdjpy"], color=BLUE, label="USD/JPY close")
ax.scatter(bought.index, bought["usdjpy"], s=70, color=ORANGE, zorder=3,
           edgecolor="white", linewidth=2, label="MoF intervention (yen bought)")
ax.set_title("Interventions sit on the day they happened, not the day MoF published them", loc="left")
ax.set_ylabel("yen per dollar")
ax.legend(frameon=False, loc="upper left")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(daily.index, daily["rate_diff"], color=BLUE, label="rate of the month (carry accrual)")
ax.plot(daily.index, daily["rate_diff_known"], color=ORANGE, label="rate known at the time (signals)")
ax.axvline(pd.Timestamp(config.JP_3M_BREAK), color="#8a8980", linestyle="--", linewidth=1)
ax.annotate("JP rate\nsource\nchanges", (pd.Timestamp(config.JP_3M_BREAK), 0.9),
            xytext=(6, 0), textcoords="offset points", ha="left", va="center", color=MUTED)
ax.set_title("US minus Japan 3-month rate: signals only see what was published", loc="left")
ax.set_ylabel("percentage points")
ax.legend(frameon=False, loc="upper center")
plt.show()

## 5. How it is checked

- **46 automated tests** (`python -m pytest -q`), including on the merged table:
  one row per trading day, prices not filled, each `_known` value equals the latest
  release before that day, delayed CFTC reports excluded, no intervention lost.
- **Every decision logged** in `docs/decisions.md` with its reason.
- **Reviewed by a teammate** through pull requests before reaching `main`.

## 6. Limitations

- Release dates are estimates from fixed rules, checked on recent releases only.
- Markets close at different times: the Nikkei closes about 15 hours before US markets.
- Gold and Brent are continuous futures and can jump when contracts roll.
- BIS revises recent quarters; data is re-fetched before final results.
- MoF interventions after the last published quarter (from July 2026) show as 0.